# Crop type classification with multi-temporal Sentinel-2 (PASTIS subset)

This notebook walks through the full workflow: data inspection, exploratory analysis, splitting, training, evaluation and visualisation. The logic lives in `src/`; the notebook calls those functions and adds commentary, so every result here can also be reproduced from the command line.

**Part 1: Data loading and exploratory data analysis**

In [ ]:
# Run from the project root so relative paths in the config resolve
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Working directory:", Path.cwd())

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loading import load_config, load_metadata, inspect_dataset
from src import eda

cfg = load_config("configs/config.yaml")
classes = {int(k): v for k, v in cfg["classes"].items()}

## 1. Data loading and inspection

Each patch has a Sentinel-2 array of shape `(T, 10, H, W)` and a label array of shape `(1, H, W)`. The inspection below checks that every image has a matching label, and reports shapes, data types, number of observations and value ranges.

In [ ]:
summary = inspect_dataset(cfg)
summary.head()

**Observations**

- 102 patches, each 128 × 128 pixels at 10 m resolution (about 1.28 km × 1.28 km).
- Every patch has exactly 46 observations with 10 bands, so all pixels share the same time axis.
- Imagery is `int16` surface reflectance scaled by 10,000. Values range from −601 to 20,716: values above 10,000 are likely clouds, snow or saturation, and negatives are atmospheric-correction artefacts. Preprocessing will clip to [0, 10,000] and rescale to [0, 1].
- No NaNs are present, but PASTIS does not mask clouds, so cloudy observations remain in the data (checked below).

## 2. Metadata: tile, dates and folds

In [ ]:
meta = load_metadata(cfg["paths"]["metadata"])
print("CRS:", meta.crs)
print("Tiles:", meta["TILE"].value_counts().to_dict())
print("Patches per fold:", meta["Fold"].value_counts().sort_index().to_dict())
dates = meta.iloc[0]["dates"]
print(f"Dates: {len(dates)} from {dates[0].date()} to {dates[-1].date()}")
meta[["TILE", "Fold", "N_Parcel", "Parcel_Cover"]].describe(include="all").round(2)

**Observations**

- All patches come from a single Sentinel-2 tile (T31TFM) in the French Lambert-93 projection (EPSG:2154), which is why they share identical acquisition dates.
- The time series runs from September 2018 to October 2019, covering one full agricultural year: sowing of winter crops in autumn, their growth and harvest in spring and summer, and the later summer crops.
- PASTIS assigns patches to 5 spatially separated folds, which we use for the train/validation/test split.
- Patches contain 21 to 82 parcels (mean about 49), and on average 70% of each patch is labelled cropland.

## 3. Full EDA pass

The function below streams through all patches once, then saves tables to `outputs/metrics/` and figures to `outputs/figures/`.

In [ ]:
res = eda.run_eda(cfg, show=True)
plt.close("all")  # figures are re-displayed individually below

### 3.1 Area of interest

*Patch footprints reprojected to WGS84 and coloured by fold. Add your interpretation of location and fold layout here.*

In [ ]:
res["figs"]["aoi"]

### 3.2 Class distribution

*Note the log scale: comment on dominant and rare classes, and on the share of background.*

In [ ]:
res["figs"]["class_distribution"]

### 3.3 Class distribution by fold

*Check whether every crop class appears in the training folds (1–3), and which classes are missing from validation (fold 4) or test (fold 5).*

In [ ]:
res["figs"]["class_by_fold"]

### 3.4 Example label maps

*Parcels are labelled as whole fields; background (black) covers roads, buildings, forest, water and unlabelled land.*

In [ ]:
res["figs"]["labels"]

### 3.5 RGB composites next to labels

*True colour (B4, B3, B2) on the clearest date, with a 2–98 percentile stretch.*

In [ ]:
res["figs"]["rgb_labels"]

### 3.6 Seasonal change

*The same patch at the clearest date in autumn, spring and summer: field colours change with crop phenology.*

In [ ]:
res["figs"]["seasonal"]

### 3.7 Cloud contamination by date

*Share of pixels with blue reflectance above 0.2, a simple proxy for cloud and haze.*

In [ ]:
res["figs"]["cloud"]

### 3.8 Temporal NDVI profiles by class

*Mean NDVI per class over the season. Winter crops should peak in spring and senesce by early summer; summer crops (corn, sunflower, soybean, sorghum) should peak in July–August.*

In [ ]:
res["figs"]["ndvi"]

### 3.9 Summary tables

In [ ]:
res["dist"].round(3)

In [ ]:
res["fold_share"].round(2)

## EDA takeaways

*To be filled in after running on the real data:* class imbalance, rare classes, fold coverage, cloudy dates, which classes have similar NDVI profiles (likely confusions), and implications for preprocessing and modelling.